# 3. Agente de Diagnóstico para Taller de Vulcanización


## Contexto del Negocio

Un taller de vulcanización atiende diariamente consultas sobre problemas en neumáticos y ruedas. Los clientes describen síntomas (vibración, pérdida de presión, desgaste irregular, etc.) y el personal debe:

1. **Diagnosticar** el problema basándose en los síntomas.
2. **Verificar** si hay stock disponible del repuesto o neumático necesario.
3. **Registrar** la solicitud de servicio para darle seguimiento.



### 1. Instalación y Configuración

In [1]:
!pip install openai python-dotenv -q

In [2]:
import os
import json
import datetime
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Carga automática del archivo .env desde el directorio actual o sus padres
for env_path in [Path.cwd() / ".env", *[parent / ".env" for parent in Path.cwd().parents]]:
    if env_path.exists():
        load_dotenv(env_path)
        break

# --- Configuración del Cliente OpenAI ---
try:
    github_token = os.environ.get("GITHUB_TOKEN")
    base_url = os.environ.get("GITHUB_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
    if not github_token:
        raise EnvironmentError("GITHUB_TOKEN no configurado. Copia .env.example a .env y agrega tu token.")

    client = OpenAI(
        base_url=base_url,
        api_key=github_token
    )
    print("✅ Cliente OpenAI configurado correctamente.")
except Exception as e:
    print(f"❌ Error configurando el cliente: {e}")
    client = None

ModuleNotFoundError: No module named 'dotenv'

### 2. Base de Datos Simulada del Taller

Antes de definir las herramientas, necesitamos datos con los que trabajar. En un proyecto real, estas consultas irían a una base de datos SQL o un sistema ERP. Aquí usamos estructuras de Python para simular ese inventario.

También creamos un registro en memoria para las solicitudes de servicio.

In [ ]:
# --- Inventario simulado del taller ---
INVENTARIO = [
    {"id": "N001", "marca": "Michelin",   "medida": "185/65 R15", "tipo": "todo terreno",  "precio": 89990,  "stock": 8},
    {"id": "N002", "marca": "Bridgestone", "medida": "195/55 R16", "tipo": "alto rendimiento", "precio": 112500, "stock": 3},
    {"id": "N003", "marca": "Pirelli",     "medida": "205/55 R16", "tipo": "turismo",      "precio": 98000,  "stock": 0},
    {"id": "N004", "marca": "Goodyear",   "medida": "175/70 R13", "tipo": "económico",    "precio": 54990,  "stock": 12},
    {"id": "N005", "marca": "Continental", "medida": "225/45 R17", "tipo": "alto rendimiento", "precio": 145000, "stock": 2},
    {"id": "N006", "marca": "Michelin",   "medida": "265/70 R16", "tipo": "4x4 / SUV",   "precio": 178000, "stock": 5},
]

# --- Registro de solicitudes (en memoria; en producción sería una BD) ---
SOLICITUDES_SERVICIO = []

# --- Catálogo de diagnósticos por síntoma ---
DIAGNOSTICOS = {
    "vibración": {
        "causa_probable": "Desbalance de ruedas o deformación del neumático (abombamiento).",
        "servicio_recomendado": "Balanceo de ruedas. Si la vibración persiste, revisar si el neumático tiene deformación interna.",
        "urgencia": "Media"
    },
    "pérdida de presión": {
        "causa_probable": "Pinchazo, válvula defectuosa o microfisuras en el neumático.",
        "servicio_recomendado": "Inspección visual y prueba de agua. Reparación de pinchazo o reemplazo de válvula según el caso.",
        "urgencia": "Alta"
    },
    "desgaste irregular": {
        "causa_probable": "Desalineación de las ruedas o presión de inflado incorrecta.",
        "servicio_recomendado": "Alineación computarizada al 4 puntos y revisión de presión de neumáticos.",
        "urgencia": "Media"
    },
    "ruido": {
        "causa_probable": "Desgaste excesivo de la banda de rodamiento, piedras incrustadas, o falla en rodamiento.",
        "servicio_recomendado": "Inspección del neumático y rodamientos. Reemplazo si el desgaste supera el indicador TWI.",
        "urgencia": "Media"
    },
    "desvío de dirección": {
        "causa_probable": "Alineación incorrecta de las ruedas o presión desigual entre neumáticos.",
        "servicio_recomendado": "Alineación al 4 puntos y nivelación de presión en todos los neumáticos.",
        "urgencia": "Alta"
    },
    "rajadura": {
        "causa_probable": "Golpe fuerte en bache, exposición prolongada al sol o neumático con exceso de años de uso.",
        "servicio_recomendado": "Reemplazo inmediato del neumático. Las rajaduras en el flanco son irreparables.",
        "urgencia": "Crítica"
    }
}

print("✅ Base de datos del taller cargada.")
print(f"   Neumáticos en catálogo: {len(INVENTARIO)}")
print(f"   Síntomas en catálogo de diagnósticos: {len(DIAGNOSTICOS)}")

### 3. Definición de las Herramientas del Agente

Creamos tres funciones de Python y sus respectivas descripciones en formato **JSON Schema** para OpenAI:

| Herramienta | Descripción |
|---|---|
| `diagnosticar_problema_neumatico` | Recibe síntomas y devuelve un diagnóstico técnico con nivel de urgencia. |
| `consultar_stock_neumaticos` | Consulta el inventario por medida, marca o tipo de neumático. |
| `registrar_solicitud_servicio` | Registra una orden de servicio con los datos del cliente y vehículo. |

In [ ]:
# ──────────────────────────────────────────────
# HERRAMIENTA 1: Diagnóstico de problemas
# ──────────────────────────────────────────────
def diagnosticar_problema_neumatico(sintoma: str) -> str:
    """
    Recibe un síntoma descrito por el cliente y retorna el diagnóstico
    técnico con el servicio recomendado y nivel de urgencia.
    """
    sintoma_lower = sintoma.lower()
    
    # Buscar coincidencia por palabra clave en el catálogo
    for clave, diagnostico in DIAGNOSTICOS.items():
        if clave in sintoma_lower:
            resultado = {
                "sintoma_detectado": clave,
                "causa_probable": diagnostico["causa_probable"],
                "servicio_recomendado": diagnostico["servicio_recomendado"],
                "urgencia": diagnostico["urgencia"]
            }
            return json.dumps(resultado, ensure_ascii=False)
    
    return json.dumps({
        "sintoma_detectado": sintoma,
        "causa_probable": "No se encontró un diagnóstico automático para este síntoma.",
        "servicio_recomendado": "Se recomienda una inspección visual presencial por un técnico del taller.",
        "urgencia": "A determinar"
    }, ensure_ascii=False)


# ──────────────────────────────────────────────
# HERRAMIENTA 2: Consulta de stock
# ──────────────────────────────────────────────
def consultar_stock_neumaticos(medida: str = None, marca: str = None, tipo: str = None) -> str:
    """
    Filtra el inventario del taller según medida, marca o tipo.
    Retorna los productos disponibles con precio y stock.
    """
    resultados = INVENTARIO.copy()

    if medida:
        resultados = [n for n in resultados if medida.lower() in n["medida"].lower()]
    if marca:
        resultados = [n for n in resultados if marca.lower() in n["marca"].lower()]
    if tipo:
        resultados = [n for n in resultados if tipo.lower() in n["tipo"].lower()]

    if not resultados:
        return json.dumps({
            "encontrados": 0,
            "mensaje": "No se encontraron neumáticos con los filtros indicados.",
            "productos": []
        }, ensure_ascii=False)

    return json.dumps({
        "encontrados": len(resultados),
        "productos": resultados
    }, ensure_ascii=False)


# ──────────────────────────────────────────────
# HERRAMIENTA 3: Registro de solicitud de servicio
# ──────────────────────────────────────────────
def registrar_solicitud_servicio(
    nombre_cliente: str,
    patente: str,
    servicio_solicitado: str,
    descripcion_problema: str = ""
) -> str:
    """
    Registra una nueva solicitud de servicio en el sistema del taller.
    """
    numero_orden = f"OS-{1000 + len(SOLICITUDES_SERVICIO) + 1}"
    solicitud = {
        "numero_orden": numero_orden,
        "fecha": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
        "nombre_cliente": nombre_cliente,
        "patente": patente.upper(),
        "servicio_solicitado": servicio_solicitado,
        "descripcion_problema": descripcion_problema,
        "estado": "Pendiente"
    }
    SOLICITUDES_SERVICIO.append(solicitud)
    
    return json.dumps({
        "exito": True,
        "mensaje": f"Solicitud registrada exitosamente con el número {numero_orden}.",
        "orden": solicitud
    }, ensure_ascii=False)


print("✅ Las 3 herramientas del agente están definidas.")

In [ ]:
# Descripción de las herramientas en formato JSON Schema para OpenAI
tools_definition = [
    {
        "type": "function",
        "function": {
            "name": "diagnosticar_problema_neumatico",
            "description": (
                "Analiza el síntoma o problema que describe el cliente sobre sus neumáticos o ruedas "
                "y devuelve un diagnóstico técnico con la causa probable, el servicio recomendado "
                "y el nivel de urgencia. Úsala cuando el cliente describe un problema como vibración, "
                "pérdida de presión, ruido, desgaste irregular, rajadura o desvío de dirección."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "sintoma": {
                        "type": "string",
                        "description": "Descripción del síntoma tal como lo relató el cliente. Ej: 'el auto vibra mucho a alta velocidad'."
                    }
                },
                "required": ["sintoma"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_stock_neumaticos",
            "description": (
                "Consulta el inventario de neumáticos disponibles en el taller. "
                "Permite filtrar por medida (ej: '185/65 R15'), marca (ej: 'Michelin') "
                "o tipo (ej: 'todo terreno', 'turismo', 'alto rendimiento', '4x4 / SUV'). "
                "Úsala cuando el cliente pregunte por disponibilidad, precios o marcas específicas."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "medida": {
                        "type": "string",
                        "description": "Medida del neumático en formato estándar, ej: '185/65 R15'. Opcional."
                    },
                    "marca": {
                        "type": "string",
                        "description": "Marca del neumático, ej: 'Bridgestone'. Opcional."
                    },
                    "tipo": {
                        "type": "string",
                        "description": "Tipo de uso del neumático: turismo, todo terreno, alto rendimiento, económico, 4x4 / SUV. Opcional."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "registrar_solicitud_servicio",
            "description": (
                "Registra una orden de servicio en el sistema del taller con los datos del cliente "
                "y el vehículo. Úsala cuando el cliente quiera agendar un servicio o cuando el "
                "diagnóstico requiera una atención presencial. Siempre solicita nombre, patente "
                "y tipo de servicio antes de registrar."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "nombre_cliente": {
                        "type": "string",
                        "description": "Nombre completo del cliente."
                    },
                    "patente": {
                        "type": "string",
                        "description": "Patente del vehículo en formato chileno, ej: 'BBCC12'."
                    },
                    "servicio_solicitado": {
                        "type": "string",
                        "description": "Nombre del servicio a realizar, ej: 'Balanceo de ruedas', 'Alineación al 4 puntos', 'Cambio de neumático'."
                    },
                    "descripcion_problema": {
                        "type": "string",
                        "description": "Descripción adicional del problema o síntoma relatado por el cliente. Opcional."
                    }
                },
                "required": ["nombre_cliente", "patente", "servicio_solicitado"]
            }
        }
    }
]

print("✅ Definición JSON Schema de las herramientas lista para la API.")

### 4. El Agente: Lógica con Múltiples Herramientas

A diferencia del notebook anterior donde el agente podía usar solo una herramienta, aquí implementamos un **bucle de herramientas**: el modelo puede necesitar llamar a múltiples funciones en una sola consulta.

Por ejemplo, si el cliente dice:
> *"Tengo vibración en mi auto. ¿Tienen neumáticos Michelin y me puede agendar?"*

El agente podría:
1. Llamar a `diagnosticar_problema_neumatico`
2. Llamar a `consultar_stock_neumaticos`
3. Llamar a `registrar_solicitud_servicio`

El bucle procesa **todos** los `tool_calls` que el modelo retorna antes de generar la respuesta final.

In [ ]:
# Mapa de nombre de función → función de Python real
HERRAMIENTAS_DISPONIBLES = {
    "diagnosticar_problema_neumatico": diagnosticar_problema_neumatico,
    "consultar_stock_neumaticos": consultar_stock_neumaticos,
    "registrar_solicitud_servicio": registrar_solicitud_servicio,
}

SYSTEM_PROMPT = """
Eres el asistente virtual de un taller de vulcanización en Chile. Tu rol es ayudar a los clientes
a diagnosticar problemas con sus neumáticos y ruedas, consultar disponibilidad de stock y
registrar solicitudes de servicio.

Directrices:
- Responde siempre en español, con un tono amable y profesional.
- Ante cualquier síntoma descrito por el cliente, SIEMPRE usa la herramienta de diagnóstico.
- Si el diagnóstico indica urgencia ALTA o CRÍTICA, advierte al cliente que no conduzca el vehículo.
- Para registrar una solicitud de servicio, pide amablemente nombre y patente si no los tienes.
- Los precios están en pesos chilenos (CLP). Preséntarlos con separador de miles (ej: $89.990).
- Si un neumático tiene stock 0, indícalo claramente y ofrece alternativas disponibles.
"""


def run_agente_vulcanizacion(user_query: str, client, historial: list = None) -> tuple[str, list]:
    """
    Ejecuta el agente para una consulta del cliente.
    Retorna la respuesta final y el historial actualizado.
    """
    if not client:
        return "Cliente no inicializado.", []

    # Inicializar historial con el system prompt si es la primera consulta
    if historial is None:
        historial = [{"role": "system", "content": SYSTEM_PROMPT}]

    historial.append({"role": "user", "content": user_query})
    print(f"\n{'='*60}")
    print(f"👤 Cliente: {user_query}")
    print(f"{'='*60}")

    # Bucle: el modelo puede llamar a múltiples herramientas
    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=historial,
            tools=tools_definition,
            tool_choice="auto",
        )

        response_message = response.choices[0].message
        historial.append(response_message)

        # Si no hay más tool_calls, el modelo ya tiene la respuesta final
        if not response_message.tool_calls:
            break

        # Procesar todos los tool_calls del turno actual
        print(f"🔧 El modelo usará {len(response_message.tool_calls)} herramienta(s)...")
        for tool_call in response_message.tool_calls:
            nombre_fn = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            print(f"   ▶ {nombre_fn}({args})")

            fn = HERRAMIENTAS_DISPONIBLES[nombre_fn]
            resultado = fn(**args)

            print(f"   ◀ Resultado: {resultado[:120]}..." if len(resultado) > 120 else f"   ◀ Resultado: {resultado}")

            historial.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": nombre_fn,
                "content": resultado,
            })

    respuesta_final = response_message.content
    print(f"\n🤖 Asistente: {respuesta_final}")
    return respuesta_final, historial


print("✅ Agente de vulcanización listo.")

### 5. Pruebas del Agente

Ejecutemos tres escenarios distintos para verificar que cada herramienta funciona correctamente.

In [ ]:
# ── ESCENARIO 1: Diagnóstico por síntoma ──────────────────────────────────
print("\n🧪 ESCENARIO 1: Cliente describe un síntoma")
historial_cliente1 = None

respuesta, historial_cliente1 = run_agente_vulcanizacion(
    "Hola, mi auto tiene una vibración muy fuerte cuando voy a más de 80 km/h.",
    client,
    historial_cliente1
)

In [ ]:
# ── ESCENARIO 2: Consulta de stock ────────────────────────────────────────
print("\n🧪 ESCENARIO 2: Cliente consulta disponibilidad de neumáticos")
historial_cliente2 = None

respuesta, historial_cliente2 = run_agente_vulcanizacion(
    "Necesito neumáticos de la medida 195/55 R16. ¿Tienen disponibles? ¿Cuánto cuestan?",
    client,
    historial_cliente2
)

In [ ]:
# ── ESCENARIO 3: Consulta compleja (diagnóstico + stock + registro) ────────
print("\n🧪 ESCENARIO 3: Consulta completa con diagnóstico, stock y registro")
historial_cliente3 = None

respuesta, historial_cliente3 = run_agente_vulcanizacion(
    "Mi neumático tiene una rajadura en el costado y necesito reemplazarlo. "
    "Me interesa ver opciones Michelin. Mi nombre es Carlos Pérez y mi patente es FGHI34.",
    client,
    historial_cliente3
)

In [ ]:
# ── Verificar las solicitudes registradas ─────────────────────────────────
print("\n📋 Solicitudes de servicio registradas hasta ahora:")
if SOLICITUDES_SERVICIO:
    for s in SOLICITUDES_SERVICIO:
        print(json.dumps(s, indent=2, ensure_ascii=False))
else:
    print("   (ninguna aún)")

### 6. Modo Conversacional

El agente también soporta conversaciones de múltiples turnos. El historial se mantiene entre llamadas, permitiendo que el cliente continúe una conversación de manera natural.

In [ ]:
# Conversación de múltiples turnos
print("🗣️  Conversación de múltiples turnos")
historial_conv = None

# Turno 1: el cliente describe el problema
_, historial_conv = run_agente_vulcanizacion(
    "Hola, noto que mi auto se va para la derecha cuando suelto el volante.",
    client, historial_conv
)

# Turno 2: el cliente pregunta por precios
_, historial_conv = run_agente_vulcanizacion(
    "¿Cuánto sale el servicio de alineación? ¿Tienen neumáticos económicos si los necesito después?",
    client, historial_conv
)

# Turno 3: el cliente quiere agendar
_, historial_conv = run_agente_vulcanizacion(
    "Quiero agendar la alineación. Soy Ana González, patente WXYZ99.",
    client, historial_conv
)

## Conclusiones

En este notebook construimos un agente de IA especializado para un taller de vulcanización usando **Function Calling** nativo de OpenAI. Los puntos clave son:

1. **Dominio específico**: Las herramientas y el system prompt están diseñados para el contexto del negocio, lo que hace que el agente sea mucho más útil que un chatbot genérico.

2. **Múltiples herramientas en un turno**: El modelo puede invocar varias funciones de forma paralela (diagnóstico + stock + registro) cuando la consulta lo requiere.

3. **Estado conversacional**: El historial de mensajes permite conversaciones naturales de múltiples turnos.

4. **Datos simulados → producción real**: El inventario y el registro de solicitudes son estructuras en memoria fácilmente reemplazables por conexiones a bases de datos reales (PostgreSQL, MongoDB, etc.).

En el próximo notebook veremos cómo **LangChain** puede simplificar aún más la gestión de este tipo de agentes con herramientas.